# Dependent-Variable Checks, Data Prep, and Extra EDA

Dependent-variable checks + data preparation + extra EDA
IPL auction price project
This is the follow-up to the general EDA. It does four things, in order:

    PART A - checks on the dependent variable (your checklist)
    PART B - verifies the redundant-column claims (prior_season, tournament)
    PART C - drops the columns we agreed on and splits the data by role
             (saves a cleaned modelling file + batsman/bowler/allrounder files)
    PART D - three extra analyses: base vs sold price, franchise spend,
             price by status

Text findings are printed AND the important tables are saved as .csv.
Figures are saved as .png. Derived data files are saved as .csv.
Everything goes to OUTPUT_DIR.

The style is deliberately plain and repetitive so every block is easy to
explain to a supervisor.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

## CONFIG  ->  change these to match your machine

In [ ]:
DATA_PATH = r"D:\DataEngineering\Final Year Project\processed_data\final_modelling_ready_dataset.csv"
OUTPUT_DIR = r"D:\DataEngineering\Final Year Project\processed_data\eda_dv_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 11


def save(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print("saved figure:", filename)

## STEP 0 - Load the data and define the analysis populations

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Full dataset shape:", df.shape)

# Price in crore, and its log (log defined only where a positive price exists)
df["price_cr"] = df["sold_price"] / 1e7
df["log_price"] = np.where(df["sold_price"] > 0, np.log(df["sold_price"]), np.nan)

# Age AT the auction, from birthdate and auction year (the scraped 'age'
# column is current age and stored as text, so we do not use it here)
birth = pd.to_datetime(df["birthdate"], errors="coerce")
df["age_at_auction"] = df["year"] - birth.dt.year

# Populations we reuse below
priced = df[df["sold_price"].notna() & (df["sold_price"] > 0)].copy()
auctioned = df[df["auction_result"].isin(["sold", "unsold"])].copy()
sold = df[df["auction_result"] == "sold"].copy()
unsold = df[df["auction_result"] == "unsold"].copy()

print("priced rows:", len(priced), "| auctioned rows:", len(auctioned),
      "| sold:", len(sold), "| unsold:", len(unsold))

## PART A - DEPENDENT-VARIABLE CHECKS

In [ ]:
print("\n==================== PART A: DV CHECKS ====================")

In [ ]:
# A1. Null values in each column (count and percent)
null_counts = df.isna().sum()
null_percent = (df.isna().mean() * 100).round(1)
null_table = pd.DataFrame({"null_count": null_counts, "null_percent": null_percent})
null_table = null_table.sort_values("null_count", ascending=False)
null_table.to_csv(os.path.join(OUTPUT_DIR, "A1_null_values.csv"))
print("\nA1. Nulls per column (top 15):")
print(null_table.head(15))

In [ ]:
# A2. Sold vs unsold counts and percentages (auction rounds only)
result_counts = auctioned["auction_result"].value_counts()
result_pct = (auctioned["auction_result"].value_counts(normalize=True) * 100).round(1)
print("\nA2. Sold vs unsold:")
print(pd.DataFrame({"count": result_counts, "percent": result_pct}))

In [ ]:
# A3. Among SOLD: how many have a sold price, and how many have a base price
sold_has_price = sold["sold_price"].notna() & (sold["sold_price"] > 0)
print("\nA3. Among sold rows (", len(sold), "):", sep="")
print("   with a sold price   :", sold_has_price.sum())
print("   WITHOUT a sold price:", (~sold_has_price).sum())
print("   with a base price   :", sold["base_price"].notna().sum())

In [ ]:
# A4. Among UNSOLD: team empty? prior-season / capped / both, yet unsold?
print("\nA4. Among unsold rows (", len(unsold), "):", sep="")
print("   team empty (NaN)               :", unsold["team"].isna().sum())
print("   team NOT empty                 :", unsold["team"].notna().sum())
print("   had a prior season, still unsold:", (unsold["has_prior_season"] == True).sum())
print("   capped, still unsold            :", (unsold["is_capped"] == True).sum())
print("   prior season AND capped, unsold :",
      ((unsold["has_prior_season"] == True) & (unsold["is_capped"] == True)).sum())

In [ ]:
# A5. Overseas vs Indian among players bought (sold): total and by year.
#     NOTE: overseas is not recorded for some early auctions, so those show
#     as 'Unknown' rather than zero.
sold["ovs"] = sold["overseas"].map({1.0: "Overseas", 0.0: "Indian"})
print("\nA5. Overseas vs Indian among sold (total):")
print(sold["ovs"].value_counts(dropna=False))

ovs_by_year = pd.crosstab(sold["year"], sold["ovs"].fillna("Unknown"))
ovs_by_year.to_csv(os.path.join(OUTPUT_DIR, "A5_overseas_by_year.csv"))
print("\nOverseas vs Indian among sold, by year:")
print(ovs_by_year)

# figure: stacked bar of Indian / Overseas / Unknown per year
fig, ax = plt.subplots(figsize=(11, 4.5))
bottom = np.zeros(len(ovs_by_year))
colours = {"Indian": "#4C72B0", "Overseas": "#DD8452", "Unknown": "#BBBBBB"}
for label in ["Indian", "Overseas", "Unknown"]:
    if label in ovs_by_year.columns:
        ax.bar(ovs_by_year.index, ovs_by_year[label], bottom=bottom,
               label=label, color=colours[label])
        bottom = bottom + ovs_by_year[label].values
ax.set_title("Overseas vs Indian players bought, by auction year")
ax.set_xlabel("Auction year")
ax.set_ylabel("Number of players bought")
ax.set_xticks(ovs_by_year.index)
ax.tick_params(axis="x", rotation=45)
ax.legend()
save(fig, "A5_overseas_by_year.png")

In [ ]:
# A6. Mega-auction years: overseas vs Indian among sold
mega_sold = sold[sold["is_mega_auction"] == True]
mega_table = pd.crosstab(mega_sold["year"], mega_sold["ovs"].fillna("Unknown"))
print("\nA6. Mega-auction years, overseas vs Indian among sold:")
print(mega_table)

In [ ]:
# A7. Sold vs unsold: how do their characteristics differ?
#     (descriptive evidence that being sold is not random)
compare_cols = ["age_at_auction", "base_price", "matches", "runs", "strikerate", "wickets"]
group_means = auctioned.groupby("auction_result")[compare_cols].mean()
group_shares = auctioned.groupby("auction_result").agg(
    share_capped=("is_capped", "mean"),
    share_overseas=("overseas", "mean"),
)
group_means.to_csv(os.path.join(OUTPUT_DIR, "A7_sold_vs_unsold_means.csv"))
group_shares.to_csv(os.path.join(OUTPUT_DIR, "A7_sold_vs_unsold_shares.csv"))
print("\nA7. Mean characteristics, sold vs unsold:")
print(group_means.round(2))
print("\nShare capped / overseas, sold vs unsold:")
print(group_shares.round(3))

## PART B - REDUNDANT-COLUMN CHECKS

In [ ]:
print("\n==================== PART B: REDUNDANCY CHECKS ====================")

In [ ]:
# B1. Is prior_season just (year - 1)? If yes, it is redundant with year.
check = df[df["prior_season"].notna()].copy()
check["prior_season_num"] = pd.to_numeric(check["prior_season"], errors="coerce")
diff = check["year"] - check["prior_season_num"]
print("\nB1. value counts of (year - prior_season):")
print(diff.value_counts(dropna=False))
print("   -> if this is all 1, prior_season carries no info beyond year.")

In [ ]:
# B2. Is tournament present exactly when has_prior_season is True?
tournament_present = df["tournament"].notna()
has_prior = df["has_prior_season"] == True
print("\nB2. tournament vs has_prior_season:")
print("   tournament non-null           :", tournament_present.sum())
print("   has_prior_season True         :", has_prior.sum())
print("   prior True but tournament NaN :", (has_prior & ~tournament_present).sum())
print("   tournament present but prior F:", (tournament_present & ~has_prior).sum())
print("   -> tournament is just prior_season as a label, so also redundant.")

In [ ]:
# B3. Show the bbi / bbm corruption, and that the numeric splits are intact.
print("\nB3. bbi / bbm are Excel-corrupted (dates), numeric splits are clean:")
print("   bbi sample :", df["bbi"].dropna().head(3).tolist())
print("   bbm sample :", df["bbm"].dropna().head(3).tolist())
print("   bbi_wkts non-null:", df["bbi_wkts"].notna().sum(),
      "| bbi_runs non-null:", df["bbi_runs"].notna().sum())

## PART C - BUILD CLEANED MODELLING DATA + SPLIT BY ROLE

In [ ]:
print("\n==================== PART C: CLEAN + SPLIT ====================")

In [ ]:
# C1. Drop the columns we agreed are corrupted or redundant.
#     NOTE: we deliberately KEEP id_source for now, because it is the
#     evidence for your data-quality / limitations section. Drop it later.
drop_now = ["bbi", "bbm", "prior_season", "tournament", "player_name_stats"]
# Optional extra drops you can decide on later (left in for now):
#   "team_stats", and the *_in_usd / *_in_cr unit-copy columns.
model_df = df.drop(columns=drop_now)
model_df.to_csv(os.path.join(OUTPUT_DIR, "C1_model_ready_cleaned.csv"), index=False)
print("C1. Dropped columns:", drop_now)
print("    cleaned frame shape:", model_df.shape)

In [ ]:
# C2. Consolidate the many role labels into four groups.
playing_role_map = {
    "Bowling Allrounder": "Allrounder",
    "Batting Allrounder": "Allrounder",
    "Allrounder": "Allrounder",
    "Bowler": "Bowler",
    "Top order Batter": "Batsman",
    "Middle order Batter": "Batsman",
    "Opening Batter": "Batsman",
    "Batter": "Batsman",
    "Wicketkeeper Batter": "Batsman",
    "Wicketkeeper": "Wicketkeeper",
}
model_df["mapped_playing_role"] = model_df["player_role"].map(playing_role_map)
print("\nC2. mapped_playing_role counts:")
print(model_df["mapped_playing_role"].value_counts(dropna=False))

In [ ]:
# C3. Create and save a file per role.
#     Here allrounders are their own file. If you would rather have them
#     appear in BOTH the batsman and bowler files, see the comment below.
batsman_df = model_df[model_df["mapped_playing_role"] == "Batsman"].copy()
bowler_df = model_df[model_df["mapped_playing_role"] == "Bowler"].copy()
allrounder_df = model_df[model_df["mapped_playing_role"] == "Allrounder"].copy()

batsman_df.to_csv(os.path.join(OUTPUT_DIR, "C3_batsman.csv"), index=False)
bowler_df.to_csv(os.path.join(OUTPUT_DIR, "C3_bowler.csv"), index=False)
allrounder_df.to_csv(os.path.join(OUTPUT_DIR, "C3_allrounder.csv"), index=False)

# To include allrounders in both role files instead, you would do:
#   batsman_with_ar = model_df[model_df["mapped_playing_role"].isin(["Batsman", "Allrounder"])]
#   bowler_with_ar  = model_df[model_df["mapped_playing_role"].isin(["Bowler", "Allrounder"])]

print("\nC3. Saved role files (rows):")
print("    batsman   :", len(batsman_df))
print("    bowler    :", len(bowler_df))
print("    allrounder:", len(allrounder_df))
# How many of each role are sold WITH a price (the real modelling sample)?
for name, sub in [("batsman", batsman_df), ("bowler", bowler_df), ("allrounder", allrounder_df)]:
    n_priced = ((sub["auction_result"] == "sold") &
                (sub["sold_price"].notna()) & (sub["sold_price"] > 0)).sum()
    print("    ", name, "sold-with-price rows:", n_priced)

## PART D - EXTRA EDA

In [ ]:
print("\n==================== PART D: EXTRA EDA ====================")

In [ ]:
# D1. Base price vs sold price (only rows that have both).
both = df[(df["base_price"].notna()) & (df["base_price"] > 0) &
          (df["sold_price"].notna()) & (df["sold_price"] > 0)].copy()
both["base_cr"] = both["base_price"] / 1e7
both["ratio"] = both["sold_price"] / both["base_price"]   # how many x over reserve

print("\nD1. Base vs sold price (", len(both), "rows with both):", sep="")
print("   correlation of log(base) and log(sold):",
      round(np.corrcoef(np.log(both["base_price"]), np.log(both["sold_price"]))[0, 1], 3))
print("   sold/base ratio summary:")
print(both["ratio"].describe().round(2))

# scatter of base vs sold price, log-log
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(both["base_cr"], both["price_cr"], s=12, alpha=0.4, color="#4C72B0")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Base price (crore, log scale)")
axes[0].set_ylabel("Sold price (crore, log scale)")
axes[0].set_title("Sold price vs base price")

# histogram of the sold/base ratio (clip the long tail so it is readable)
clipped_ratio = both["ratio"].clip(upper=20)
axes[1].hist(clipped_ratio, bins=40, color="#55A868", edgecolor="white")
axes[1].set_xlabel("Sold price / base price  (clipped at 20x)")
axes[1].set_ylabel("Number of players")
axes[1].set_title("How far over reserve players go")
save(fig, "D1_base_vs_sold_price.png")

In [ ]:
# D2. Franchise-level spend among sold players with a price.
#     NOTE: team names vary across eras (e.g. Kings XI Punjab -> Punjab Kings),
#     so totals may be split across a franchise's different names.
sold_priced = sold[sold["sold_price"].notna() & (sold["sold_price"] > 0)].copy()
team_spend = sold_priced.groupby("team")["price_cr"].agg(["sum", "mean", "count"])
team_spend = team_spend.sort_values("sum", ascending=False)
team_spend.to_csv(os.path.join(OUTPUT_DIR, "D2_franchise_spend.csv"))
print("\nD2. Top 10 franchises by total spend (crore):")
print(team_spend.head(10).round(2))

top_teams = team_spend.head(12)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_teams.index[::-1], top_teams["sum"].values[::-1], color="#4C72B0")
ax.set_title("Total spend by franchise (top 12)")
ax.set_xlabel("Total spend (crore, all auctions)")
save(fig, "D2_franchise_spend.png")

In [ ]:
# D3. Price by status (New / Retained / RTM).
status_priced = priced[priced["status"].notna()].copy()
status_summary = status_priced.groupby("status")["price_cr"].agg(["count", "median", "mean"])
status_summary.to_csv(os.path.join(OUTPUT_DIR, "D3_price_by_status.csv"))
print("\nD3. Price by status (crore):")
print(status_summary.round(2))

status_order = ["New", "Retained", "RTM"]
status_data = []
status_labels = []
for s in status_order:
    values = status_priced.loc[status_priced["status"] == s, "log_price"].dropna().values
    if len(values) > 0:
        status_data.append(values)
        status_labels.append(s)

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(status_data)
ax.set_xticks(range(1, len(status_labels) + 1))
ax.set_xticklabels(status_labels)
ax.set_title("Log price by status")
ax.set_ylabel("log(sold price)")
save(fig, "D3_price_by_status.png")

print("\nDONE. All tables, figures, and derived files are in:", OUTPUT_DIR)